# StoryHub Colab Worker — Trạng thái trực tiếp

Notebook: cài repo → nhập URL/token → xác thực → chạy worker với thanh tiến trình, log và tình trạng chương/mẫu.


In [ ]:
#@title 1. Cài repository từ GitHub
REPO_URL = "https://github.com/USERNAME/storyhub-colab-worker.git" #@param {type:"string"}
import os, subprocess, sys
from urllib.parse import urlsplit

repo = REPO_URL.strip()
parsed = urlsplit(repo)
if parsed.scheme != 'https' or parsed.netloc.lower() != 'github.com' or not parsed.path.endswith('.git'):
    raise ValueError('Hãy nhập GitHub clone URL dạng https://github.com/USERNAME/REPO.git')
workdir = '/content/storyhub-colab-worker'
if os.path.isdir(os.path.join(workdir,'.git')):
    current = subprocess.check_output(['git','-C',workdir,'remote','get-url','origin'],text=True).strip()
    if current != repo:
        raise RuntimeError('Đã có repo khác tại /content/storyhub-colab-worker. Hãy đổi runtime hoặc xóa repo cũ có chủ đích.')
    subprocess.run(['git','-C',workdir,'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','--depth','1',repo,workdir],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',workdir+'/requirements.txt'],check=True)
print('✓ Đã cài mã nguồn. Chạy cell 2 để kết nối Hub.')


In [ ]:
#@title 2. Nhập địa chỉ Hub và cấu hình AI
HUB_URL = "https://hub.example.com" #@param {type:"string"}
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct" #@param ["Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen3-1.7B"]
ENGINE = "transformers" #@param ["transformers", "mock"]

import getpass, os
HUB_URL = HUB_URL.strip().rstrip('/')
if not HUB_URL.startswith('https://'):
    raise ValueError('Colab yêu cầu URL Hub HTTPS công khai; không dùng localhost.')
WORKER_TOKEN = getpass.getpass('Nhập Worker Token từ Hub (ẩn ký tự): ').strip()
if not WORKER_TOKEN:
    raise ValueError('Worker Token không được để trống')
print('✓ Đã nhập địa chỉ Hub và token (không hiển thị token).')


In [ ]:
#@title 3. Kiểm tra kết nối Hub và phần cứng
import sys, os
sys.path.insert(0, '/content/storyhub-colab-worker')
from check_connection import check
status = check(HUB_URL, WORKER_TOKEN)
print('✓ Hub xác thực thành công:', status['host'], '| Rules hash:', status['rules_hash'])
try:
    import torch
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
except Exception as ex:
    print('Không đọc được thông tin GPU:', str(ex))


In [ ]:
#@title 4. Chạy Worker và xem tiến trình trực tiếp
# Chạy cell này liên tục. Khi Colab sẵn sàng, nhấn START trên Hub.
import os, sys, json, time, subprocess, threading
from IPython.display import display
import ipywidgets as widgets

state_file='/content/storyhub-worker-status.json'
try: os.unlink(state_file)
except FileNotFoundError: pass
settings=dict(os.environ,HUB_URL=HUB_URL,WORKER_TOKEN=WORKER_TOKEN,MODEL_ID=MODEL_ID,ENGINE=ENGINE,STORYHUB_STATUS_FILE=state_file)
bar=widgets.IntProgress(value=0,min=0,max=100,description='Chương:',bar_style='info',layout=widgets.Layout(width='98%'))
status=widgets.HTML(value='<b>Khởi động...</b>')
info=widgets.HTML(value='')
log=widgets.Output(layout={'border':'1px solid #dce5f3','height':'230px','overflow_y':'auto'})
display(widgets.VBox([widgets.HTML('<h3>StoryHub Colab — Giám sát worker</h3>'),status,bar,info,log]))
stop_ui=threading.Event()

def update_ui():
    while not stop_ui.is_set():
        try:
            if os.path.exists(state_file):
                with open(state_file,encoding='utf-8') as f: d=json.load(f)
                done=int(d.get('chapter') or 0); total=int(d.get('total_chapters') or 0)
                bar.value=min(100,round(done/total*100)) if total else 0
                status.value='<b>Trạng thái:</b> '+str(d.get('stage','—'))+' &nbsp; | &nbsp; <b>GPU:</b> '+str(d.get('gpu','—'))
                from html import escape
                info.value='<b>Worker:</b> '+escape(str(d.get('worker_id','—')))+'<br><b>Truyện:</b> '+escape(str(d.get('story_title') or d.get('story_id') or 'Đang chờ Hub START'))+'<br><b>Chương:</b> '+str(done)+' / '+str(total)+' &nbsp; | &nbsp; <b>SFT mới:</b> '+str(d.get('samples',0))+'<br><b>Lỗi:</b> '+escape(str(d.get('error','')))
        except Exception: pass
        stop_ui.wait(2)
thread=threading.Thread(target=update_ui,daemon=True)
thread.start()
p=None
try:
    p=subprocess.Popen([sys.executable,'-u','/content/storyhub-colab-worker/worker.py'],env=settings,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        with log: print(line.rstrip())
    if p.wait()!=0:print('Worker kết thúc với mã lỗi:',p.returncode)
except KeyboardInterrupt:
    print('Đang dừng worker...')
finally:
    if p and p.poll() is None:
        p.terminate()
        try:p.wait(timeout=10)
        except subprocess.TimeoutExpired:p.kill()
    stop_ui.set();thread.join(timeout=3)
    settings.pop('WORKER_TOKEN',None)


## Khi gặp sự cố

- **401/403:** Token phải là Worker Token của Hub, không phải Admin Token.
- **Connection error:** Hub cần URL HTTPS có thể truy cập từ Colab (thường qua Cloudflare Tunnel).
- **Idle nhưng không có truyện:** Bạn chưa nhấn START, hoặc chưa có truyện `source_ready`.
- **Hết VRAM:** Chọn model 1.5B hoặc thay runtime GPU; model 3B phụ thuộc VRAM thực tế.
- **Colab mất phiên:** Mở notebook chạy lại; Hub cấp lại job sau khi lease cũ hết hạn và sử dụng checkpoint đã được xác nhận.

**Lưu ý:** Không lưu notebook sau khi nhập token nếu notebook đã được chỉnh sửa để hiển thị token ra ô mã. Không commit khóa bí mật lên GitHub.
